# Vehicle Insurance Claim Guide
## RAG-Based Document Question Answering System

**Course/Project:** Generative AI / RAG  
**Application:** Motor Vehicle Insurance Claim Assistance

### Objective
Build a Retrieval-Augmented Generation (RAG) system over motor insurance policy and claim-process documents. The system retrieves relevant document chunks and generates grounded answers, including a quoted policy exclusion for an excluded scenario such as driving under the influence of alcohol.

## 1. Problem Statement

Policyholders may find motor insurance claim procedures difficult to understand after an accident. Important information such as reporting steps, required documents, claim settlement procedures, and exclusions may be distributed across multiple policy documents.

This project develops a **Document QA system** that allows a user to ask natural-language questions and receive answers grounded in the supplied motor insurance documents.

## 2. System Architecture

```text
                 MOTOR INSURANCE PDFs
                         |
                         v
                  PDF Text Extraction
                       PyPDF
                         |
                         v
                     Chunking
                  800 / 100 overlap
                         |
                         v
             Gemini Embedding 001
              RETRIEVAL_DOCUMENT
                         |
                         v
                  768-D Vectors
                         |
                         v
                    FAISS Index
                         ^
                         |
                    User Query
                         |
                         v
             Gemini Embedding 001
               RETRIEVAL_QUERY
                         |
                         v
                   FAISS Top-K
                     K = 4
                         |
                         v
                Retrieved Chunks
                         |
                         v
                  Gemini Flash
                         |
                         v
               Grounded Answer
                  + Source/Page
```

### RAG Workflow

1. Load the insurance PDFs.
2. Extract text using PyPDF.
3. Split text into overlapping chunks.
4. Generate Gemini document embeddings.
5. Store embeddings in FAISS.
6. Convert the user's question into a retrieval-query embedding.
7. Retrieve the top 4 relevant chunks.
8. Pass the retrieved context to Gemini.
9. Generate a grounded answer with source and page information.
10. For exclusions, quote the relevant policy wording.

## 3. Document Corpus

The system uses the following motor-insurance documents:

| # | Document |
|---|---|
| 1 | Accident Claim Process.pdf |
| 2 | Car Insurance.pdf |
| 3 | Motor Insurance Claim Process.pdf |
| 4 | Motor_Claim_Form.pdf |
| 5 | motor_ots_claim.pdf |
| 6 | motor_policy_exclusions.pdf |

The documents were standardized into text-readable PDFs where required so that the RAG ingestion pipeline could reliably extract their text.

## 4. Preprocessing and Chunking

### Text extraction
**Library:** `pypdf`

The PDF pages are read and their text is extracted before indexing.

### Chunking configuration

- **Chunk size:** 800 characters
- **Chunk overlap:** 100 characters
- **Purpose:** Preserve context across chunk boundaries while keeping retrieved passages manageable.

### Example

```text
Document
   |
   +---- Chunk 1: characters 0-800
   |
   +---- Chunk 2: characters 700-1500
   |
   +---- Chunk 3: characters 1400-2200
```

The overlap helps prevent important information from being split between two unrelated chunks.

## 5. Embedding Model

### Model
**Gemini Embedding 001**

Two retrieval-specific task types are used:

- `RETRIEVAL_DOCUMENT` — for insurance-document chunks
- `RETRIEVAL_QUERY` — for user questions

### Vector dimension

**768 dimensions**

The final FAISS database contains the same-dimensional vectors for both documents and queries.

### Why embeddings?

Embeddings represent text as numerical vectors so that semantically similar questions and document passages can be retrieved even when they do not use exactly the same words.

## 6. Vector Database

### Vector store
**FAISS (Facebook AI Similarity Search)**

### Configuration

- Index: `IndexFlatL2`
- Vectors: generated from Gemini Embedding 001
- Retrieval: similarity search
- **Top-K:** 4

The final build used:

- **Documents:** 6
- **Chunks:** 184
- **Vectors:** 184
- **Embedding dimension:** 768

The generated files are:

```text
vectorstore/
├── insurance.index
├── chunks.pkl
└── config.pkl
```

## 7. Generation Model and Prompting

The retrieved document chunks are supplied to the Gemini generation model.

### Core grounding instructions

```text
Answer only using the retrieved insurance documents.

Do not use outside knowledge.

Do not invent information.

If the information is not present in the retrieved
context, say that it could not be found in the
provided insurance documents.

For claim-process questions, provide numbered steps.

For exclusion questions, clearly identify the exclusion
and provide an exact short quotation from the retrieved
document.

Never invent a quotation.

Always provide the source document and page number.
```

This prompt is designed to reduce hallucination and make the answer traceable to the source documents.

## 8. Streamlit User Interface

The Streamlit application provides:

- Natural-language question input
- Predefined demonstration questions
- RAG retrieval
- Gemini answer generation
- Retrieved-chunk display
- Source document and page information
- Exclusion-query warning
- Out-of-scope handling
- Academic prototype disclaimer

The application is launched using:

```powershell
streamlit run app.py
```

## 9. Required Demonstration Queries

### Query 1 — Normal answerable query

**Question:**

> What is the first step after a car accident?

**Expected behavior:** Retrieve the relevant accident/claim-process information and answer using the insurance documents.

**Execution screenshot:**

### Query 2 — Required documents

**Question:**

> What documents are required to file a motor insurance claim?

**Expected behavior:** Retrieve the relevant claim-form and claim-process passages and list the required documents based on the corpus.

**Execution screenshot:**

### Query 3 — Multi-chunk step-by-step query

**Question:**

> Explain the complete claim process from the accident until settlement.

**Expected behavior:** Retrieve information from multiple relevant chunks and combine it into a coherent, numbered claim workflow.

**Execution screenshot:**

### Query 4 — Excluded scenario ⭐

**Question:**

> Will my motor insurance claim be covered if I was driving under the influence of alcohol?

**Expected behavior:**

1. Identify the relevant policy exclusion.
2. Quote the actual wording from the retrieved policy context.
3. Show the source document and page.
4. Avoid inventing policy wording.

**Important:** The final submitted screenshot should visibly show the **actual quotation from `motor_policy_exclusions.pdf`** and its source/page information.

**Execution screenshot:**

### Query 5 — Out-of-scope / unsupported question

**Question:**

> What is the premium for a Mercedes-Benz C-Class in 2027?

**Expected behavior:** The system should not invent a price. It should state that the information cannot be found in the provided insurance documents.

**Execution screenshot:**

## 10. Evaluation of the RAG System

The system demonstrates four important RAG behaviors:

| Capability | Demonstration |
|---|---|
| Document-grounded QA | Normal insurance questions |
| Step-by-step reasoning from retrieved context | Complete claim process |
| Multi-chunk retrieval | Claim process spanning multiple documents/chunks |
| Exclusion handling | Drunk-driving/alcohol query with quotation |
| Hallucination control | Unsupported vehicle-premium query |
| Evidence traceability | Retrieved chunks + source + page |

The system does not make an official insurance claim decision. It is an academic document-grounded QA prototype.

## 11. Advantages

1. **Natural-language interaction** — Users can ask questions conversationally.
2. **Grounded answers** — Responses are based on retrieved insurance documents.
3. **Semantic retrieval** — Embeddings support meaning-based search.
4. **Evidence visibility** — Retrieved chunks, source documents, and pages are displayed.
5. **Exclusion awareness** — Policy exclusions can be explicitly highlighted and quoted.
6. **Hallucination control** — Unsupported questions are rejected instead of being answered from outside knowledge.
7. **Modular architecture** — Document ingestion, retrieval, generation, and UI are separated.

## 12. Limitations

1. The system depends on the quality and completeness of the supplied documents.
2. Incorrect or outdated policy documents can lead to incorrect retrieved information.
3. OCR/scanned-document quality can affect extraction accuracy if source PDFs are image-based.
4. Vector similarity retrieval may occasionally return less relevant chunks.
5. The generated answer still needs verification against the original policy for real-world decisions.
6. The prototype should not be treated as an official insurance claim approval system.
7. API availability, quotas, and network connectivity can affect Gemini-based processing.

## 13. Technology Stack

| Component | Technology |
|---|---|
| Programming language | Python |
| PDF extraction | PyPDF |
| Embeddings | Gemini Embedding 001 |
| Vector database | FAISS |
| Generation | Gemini Flash |
| UI | Streamlit |
| Environment configuration | python-dotenv |
| Numerical processing | NumPy |
| API communication | Requests |

### Main project files

```text
Vehicle_Insurance_RAG/
├── data/
├── vectorstore/
├── app.py
├── build_rag.py
├── requirements.txt
└── Vehicle_Insurance_RAG.ipynb
```

## 14. How to Run

### Install dependencies

```powershell
python -m pip install -r requirements.txt
```

### Configure the API key

Create a `.env` file:

```text
GEMINI_API_KEY=YOUR_API_KEY
```

**Do not submit the `.env` file or expose the API key.**

### Build the vector database

```powershell
python build_rag.py
```

### Run the application

```powershell
streamlit run app.py
```

## 15. Conclusion

The Vehicle Insurance Claim Guide demonstrates a practical Retrieval-Augmented Generation pipeline for insurance document question answering.

The system combines **PyPDF**, **Gemini Embedding 001**, **FAISS**, **Gemini generation**, and **Streamlit** to provide document-grounded responses.

The most important demonstration is the excluded-scenario query, where the system retrieves the relevant policy exclusion and quotes the policy wording rather than relying on unsupported general knowledge.

The project therefore provides a practical prototype for helping policyholders understand claim procedures while maintaining source traceability and reducing hallucination.